# Pipeline dự đoán review xấu trên Olist

**Bài toán.** Ngay khi một đơn vừa giao xong, dự đoán khách có cho review 1–2 sao không, để đội CSKH
liên hệ trước. Thời điểm dự đoán là `order_delivered_customer_date`: mọi feature phải biết được
**tại lúc đó**.

**Chia dữ liệu** theo ngày mua (định nghĩa chung của track): train trước 2018-01-01, validation
2018-01-01 tới trước 2018-04-01, test 2018-04-01 tới hết 2018-08-31.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hằng hoặc hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, in kết quả. Test bỏ qua chúng.
- Cell `load_olist_table` đã viết sẵn: đọc hiểu, không cần sửa.
- Chấm: `uv run pytest tracks/data-science/modules/10-feature-engineering/exercises/intermediate/02-olist-pipeline -q`

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

In [ ]:
# Cell viết sẵn: đọc 8 bảng Olist, tạo một dòng cho mỗi đơn giao xong có review.
# Thời điểm dự đoán: lúc đơn vừa giao xong (order_delivered_customer_date), trước khi khách review.
DATES = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
         "order_delivered_customer_date", "order_estimated_delivery_date"]
ZIP = {"customer_zip_code_prefix": str, "seller_zip_code_prefix": str, "geolocation_zip_code_prefix": str}


def _haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = (np.radians(v) for v in (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


def load_olist_table(raw) -> pd.DataFrame:
    raw = Path(raw)
    orders = pd.read_csv(raw / "olist_orders_dataset.csv", parse_dates=DATES)
    reviews = pd.read_csv(raw / "olist_order_reviews_dataset.csv",
                          parse_dates=["review_creation_date", "review_answer_timestamp"])
    customers = pd.read_csv(raw / "olist_customers_dataset.csv", dtype=ZIP)
    items = pd.read_csv(raw / "olist_order_items_dataset.csv")
    products = pd.read_csv(raw / "olist_products_dataset.csv")
    sellers = pd.read_csv(raw / "olist_sellers_dataset.csv", dtype=ZIP)
    payments = pd.read_csv(raw / "olist_order_payments_dataset.csv")
    geo = pd.read_csv(raw / "olist_geolocation_dataset.csv", dtype=ZIP)

    delivered = orders[(orders.order_status == "delivered") & orders.order_delivered_customer_date.notna()]
    latest = reviews.sort_values("review_answer_timestamp").drop_duplicates("order_id", keep="last")
    df = delivered.merge(latest, on="order_id").merge(customers, on="customer_id")
    df["bad_review"] = (df.review_score <= 2).astype(int)

    day = pd.Timedelta(days=1)
    df["purchase_hour"] = df.order_purchase_timestamp.dt.hour
    df["purchase_dow"] = df.order_purchase_timestamp.dt.day_of_week
    df["promised_days"] = (df.order_estimated_delivery_date - df.order_purchase_timestamp) / day
    df["delivery_days"] = (df.order_delivered_customer_date - df.order_purchase_timestamp) / day
    df["delay_days"] = (df.order_delivered_customer_date - df.order_estimated_delivery_date) / day
    df["approve_hours"] = (df.order_approved_at - df.order_purchase_timestamp) / pd.Timedelta(hours=1)

    items = items.merge(products, on="product_id", how="left").merge(sellers, on="seller_id", how="left")
    per_order = items.groupby("order_id").agg(
        n_items=("order_item_id", "size"), n_sellers=("seller_id", "nunique"),
        price=("price", "sum"), freight=("freight_value", "sum"), weight_g=("product_weight_g", "sum"),
        photos=("product_photos_qty", "mean"), desc_len=("product_description_lenght", "mean"))
    first = items[items.order_item_id == 1][
        ["order_id", "seller_id", "product_category_name", "seller_state", "seller_zip_code_prefix"]]
    df = df.merge(per_order, on="order_id").merge(first, on="order_id")
    df["freight_ratio"] = df.freight / (df.price + df.freight)
    main_payment = payments.sort_values("payment_value", ascending=False).drop_duplicates("order_id")
    df = df.merge(main_payment[["order_id", "payment_type", "payment_installments"]], on="order_id", how="left")

    ok = geo.geolocation_lat.between(-34, 6) & geo.geolocation_lng.between(-74, -34)
    centroid = geo[ok].groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]].median()
    c = centroid.reindex(df.customer_zip_code_prefix).to_numpy()
    s = centroid.reindex(df.seller_zip_code_prefix).to_numpy()
    df["distance_km"] = _haversine_km(c[:, 0], c[:, 1], s[:, 0], s[:, 1])
    df["same_state"] = (df.customer_state == df.seller_state).astype(int)
    return df.sort_values("order_purchase_timestamp", kind="stable").reset_index(drop=True)


def time_split(df: pd.DataFrame) -> dict:
    t = df.order_purchase_timestamp
    return {"train": df[t < "2018-01-01"],
            "val": df[(t >= "2018-01-01") & (t < "2018-04-01")],
            "test": df[(t >= "2018-04-01") & (t < "2018-09-01")]}

In [ ]:
# explore: tìm gốc repo và đọc bảng feature Olist (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
table = load_olist_table(ROOT / "data" / "raw")
splits = time_split(table)
train, val, test = splits["train"], splits["val"], splits["test"]
print(table.shape, {k: len(v) for k, v in splits.items()})

## 1. Cột nào là leakage?

Bảng có cả cột về đơn, sản phẩm, seller, khách **và** cột về review. Với từng cột, hỏi: *giá trị
này được ghi nhận lúc nào, so với lúc đơn vừa giao?* Cột ghi nhận sau thời điểm dự đoán, hoặc suy
ra từ chính nhãn, là leakage. Cẩn thận với cột có *giá trị* là một ngày tương lai nhưng được *ghi
nhận* từ lúc mua (ngày hẹn giao): cột đó hợp lệ.

In [ ]:
# explore: các cột của bảng và thử "AUC một mình" của vài cột số đáng ngờ
print(list(table.columns))
y_all = table.bad_review
for col in ["delay_days", "delivery_days", "review_score"]:
    auc = roc_auc_score(y_all, table[col])
    print(f"{col:15s} {max(auc, 1 - auc):.3f}")
comment_len = table.review_comment_message.fillna("").str.len()
auc = roc_auc_score(y_all, comment_len)
print(f"{'comment_len':15s} {max(auc, 1 - auc):.3f}")

In [ ]:
# Liệt kê mọi cột của bảng KHÔNG được dùng làm feature vì là leakage (kể cả chính nhãn).
LEAKY_COLUMNS = []  # TODO

## 2. Chọn feature

Chia feature hợp lệ thành ba nhóm theo cách mã hoá: số (impute + scale), phân loại ít giá trị
(one-hot), phân loại nhiều giá trị như `seller_id` (target encoding có cross fitting). Không dùng cột
id thuần (`order_id`, `customer_id`) hay timestamp thô.

In [ ]:
NUMERIC = []           # TODO: cột số hợp lệ
CATEGORICAL = []       # TODO: cột phân loại ít giá trị
HIGH_CARDINALITY = []  # TODO: cột phân loại rất nhiều giá trị

## 3. Pipeline

`build_pipeline` trả một `Pipeline` **chưa fit**: `ColumnTransformer` (số: impute median rồi
`StandardScaler`; phân loại: impute hằng rồi `OneHotEncoder` chịu được category lạ; nhiều giá trị:
`TargetEncoder`) nối với `LogisticRegression(C=C)`. Nhóm nào rỗng thì bỏ nhóm đó.

`fit_and_score` fit pipeline trên `train` (nhãn là cột `target`) và trả ROC-AUC trên `val`.

In [ ]:
def build_pipeline(numeric: list[str], categorical: list[str], high_card: list[str],
                   C: float = 1.0) -> Pipeline:
    # TODO: ColumnTransformer với tối đa 3 nhánh, rồi LogisticRegression(C=C, max_iter=2000)
    raise NotImplementedError


def fit_and_score(pipe: Pipeline, train: pd.DataFrame, val: pd.DataFrame, target: str = "bad_review") -> float:
    # TODO: fit trên train, trả ROC-AUC trên val
    raise NotImplementedError

In [ ]:
# explore: điểm trên validation, và điều gì xảy ra nếu lỡ thêm một feature leakage
assert not set(NUMERIC + CATEGORICAL + HIGH_CARDINALITY) & set(LEAKY_COLUMNS)
pipe = build_pipeline(NUMERIC, CATEGORICAL, HIGH_CARDINALITY)
print("AUC val:", round(fit_and_score(pipe, train, val), 4))
print("không có seller_id:", round(fit_and_score(build_pipeline(NUMERIC, CATEGORICAL, []), train, val), 4))
leaky_train = train.assign(comment_len=train.review_comment_message.fillna("").str.len())
leaky_val = val.assign(comment_len=val.review_comment_message.fillna("").str.len())
print("thêm comment_len (leakage):", round(fit_and_score(
    build_pipeline(NUMERIC + ["comment_len"], CATEGORICAL, HIGH_CARDINALITY), leaky_train, leaky_val), 4))

In [ ]:
# explore: hệ số lớn nhất. Cột số đã scale nên so được với nhau; one-hot (0/1) và target encoding
# (thang xác suất) có thang khác, chỉ nên so trong cùng nhóm
pipe = build_pipeline(NUMERIC, CATEGORICAL, HIGH_CARDINALITY)
fit_and_score(pipe, train, val)
names = pipe[0].get_feature_names_out()
coef = pd.Series(pipe[-1].coef_[0], index=names)
coef.reindex(coef.abs().sort_values(ascending=False).index).head(12)

## 4. Nhìn test đúng một lần

Chỉ khi đã chốt feature và tham số trên validation mới fit lại trên train + validation và đo trên
test. So hai con số: vì sao AUC test thấp hơn AUC validation khá nhiều? (Gợi ý: tỉ lệ giao trễ và
tỉ lệ review xấu ở mỗi giai đoạn.)

In [ ]:
# explore: fit lại trên train + val, đo trên test; so tỉ lệ giao trễ từng giai đoạn
dev = pd.concat([train, val])
pipe = build_pipeline(NUMERIC, CATEGORICAL, HIGH_CARDINALITY)
print("AUC test:", round(fit_and_score(pipe, dev, test), 4))
for name, part in splits.items():
    print(name, "trễ:", round((part.delay_days > 0).mean(), 3), "review xấu:", round(part.bad_review.mean(), 3))